In [1]:
!pip install -q sentence-transformers datasets pandas

In [2]:
# %% Cell 2: load models (base + ours). If the fine-tuned folder is missing, upload the zip first.

import os, glob, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

print("GPU available:", torch.cuda.is_available())   # must be True, or encoding will take hours

if not os.path.isdir("scibot-sci-retriever"):
    from google.colab import files
    print("Upload scibot-sci-retriever.zip (the file you downloaded earlier)")
    up = files.upload()
    zipname = list(up.keys())[0]
    os.system(f'unzip -q -o "{zipname}"')
assert os.path.exists("scibot-sci-retriever/modules.json"), "modules.json not found: check the unzip/nesting"

BASE = "BAAI/bge-small-en-v1.5"
QP = "Represent this sentence for searching relevant passages: "   # same query prefix used in training
base_model = SentenceTransformer(BASE)
ours = SentenceTransformer("scibot-sci-retriever")
for m in (base_model, ours):
    m.max_seq_length = 256      # same truncation for both models, keeps big corpora fast

GPU available: True
Upload scibot-sci-retriever.zip (the file you downloaded earlier)


Saving scibot-sci-retriever.zip to scibot-sci-retriever.zip


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
# %% Cell 3: helpers (same metrics as before: Recall@5, MRR@10, nDCG@10, binary relevance)

from datasets import load_dataset

def metrics_for(rankings, qids, rel):
    rec5, mrr, ndcg = [], [], []
    for qid in qids:
        r = rel[qid]; ranked = rankings[qid][:10]
        rec5.append(len(set(ranked[:5]) & r) / len(r))
        mrr.append(next((1.0 / (i + 1) for i, d in enumerate(ranked) if d in r), 0.0))
        dcg = sum(1.0 / np.log2(i + 2) for i, d in enumerate(ranked) if d in r)
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(r), 10)))
        ndcg.append(dcg / idcg)
    return {"Recall@5": np.mean(rec5), "MRR@10": np.mean(mrr), "nDCG@10": np.mean(ndcg)}

def load_beir(name):
    corpus_ds = load_dataset(f"BeIR/{name}", "corpus")["corpus"]
    queries_ds = load_dataset(f"BeIR/{name}", "queries")["queries"]
    qrels = load_dataset(f"BeIR/{name}-qrels")["test"]
    corpus = {str(r["_id"]): (r["title"] + ". " + r["text"]).strip() for r in corpus_ds}
    queries = {str(r["_id"]): r["text"] for r in queries_ds}
    rel = {}
    for r in qrels:
        if r["score"] > 0:                       # keep only relevant documents
            rel.setdefault(str(r["query-id"]), set()).add(str(r["corpus-id"]))
    qids = [q for q in rel if q in queries]
    return corpus, queries, rel, qids

def eval_model(model, d_ids, d_emb_cache, qids, queries, rel, key):
    if key not in d_emb_cache:
        d_emb_cache[key] = model.encode([corpus_texts[i] for i in range(len(d_ids))],
                                        batch_size=128, normalize_embeddings=True, show_progress_bar=True)
    q_emb = model.encode([QP + queries[q] for q in qids], normalize_embeddings=True)
    sims = q_emb @ d_emb_cache[key].T
    rankings = {q: [d_ids[i] for i in np.argpartition(-sims[k], 10)[:10][np.argsort(-sims[k][np.argpartition(-sims[k], 10)[:10]])]]
                for k, q in enumerate(qids)}
    return metrics_for(rankings, qids, rel)

In [4]:
# %% Cell 4: run both benchmarks (SCIDOCS ~25k docs, TREC-COVID ~171k docs; expect 10-25 minutes on a T4)

all_results = {}
for ds_name in ["scidocs", "trec-covid"]:
    print(f"\n=== {ds_name} ===")
    corpus, queries, rel, qids = load_beir(ds_name)
    d_ids = list(corpus.keys()); corpus_texts = [corpus[d] for d in d_ids]
    print(f"docs={len(d_ids)} test_queries={len(qids)}")
    cache = {}
    all_results[ds_name] = {
        "Off-the-shelf BGE-small": eval_model(base_model, d_ids, cache, qids, queries, rel, "base"),
        "Fine-tuned (ours)":       eval_model(ours,       d_ids, cache, qids, queries, rel, "ours"),
    }
    print(pd.DataFrame(all_results[ds_name]).T.round(3))
    del cache; torch.cuda.empty_cache()


=== scidocs ===


README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.7MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating corpus split:   0%|          | 0/25657 [00:00<?, ? examples/s]

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 92.6kB            

... (19 outputs left out)

docs=171332 test_queries=50


Batches:   0%|          | 0/1339 [00:00<?, ?it/s]

Batches:   0%|          | 0/1339 [00:00<?, ?it/s]

                         Recall@5  MRR@10  nDCG@10
Off-the-shelf BGE-small     0.011   0.945    0.798
Fine-tuned (ours)           0.010   0.869    0.744


In [5]:
# %% Cell 5: save and download

rows = []
for ds_name, res in all_results.items():
    for model_name, m in res.items():
        rows.append({"dataset": ds_name, "model": model_name, **{k: round(float(v), 4) for k, v in m.items()}})
out = pd.DataFrame(rows)
print(out)
out.to_csv("scidocs_trec_covid_results.csv", index=False)
from google.colab import files
files.download("scidocs_trec_covid_results.csv")

      dataset                    model  Recall@5  MRR@10  nDCG@10
0     scidocs  Off-the-shelf BGE-small    0.1516  0.3500   0.2058
1     scidocs        Fine-tuned (ours)    0.1496  0.3382   0.2011
2  trec-covid  Off-the-shelf BGE-small    0.0109  0.9450   0.7982
3  trec-covid        Fine-tuned (ours)    0.0102  0.8689   0.7443


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>